# Alert Mesh — how a warning reaches people with no signal

Alert Mesh is an emergency warning app for floods and bushfires in remote Australia.
Official warnings are **signed**, sent to phones that have a connection, and then
**passed from phone to phone over Bluetooth**, so they also reach people with no signal.
People can send back an **SOS**, an **"I'm safe"** or a **hazard report** the same way.

This notebook runs the same rules as the iPhone app, ported to Python in `alertmesh/`.
The warning bytes match the app's byte for byte (see `tests/`). What the notebook adds
is a simulation: phones on a map, passing warnings along.

**How to run:** open this file in VS Code and choose **Run All**. It takes about a minute.
Every run gives the same numbers, because the simulations use fixed random seeds.

*Simulation, not field data.* Bluetooth is modelled as a plain 60 m range with no signal
loss, and the internet as instant. Real Bluetooth reach varies from about 10 m to over 100 m.

In [ ]:
import pandas as pd

from alertmesh import geohash, metrics, places, proximity, reports, wire
from alertmesh.alert_store import AlertStore
from alertmesh.mesh_sim import COMMUNITY_REPORT_TYPE, OFFICIAL_ALERT_TYPE, Mesh, offset_m
from alertmesh.signer import OfficialAlertSigner, WarningDraft, new_alert_id
from alertmesh.wire import HazardType, Severity

START = metrics.START_MS            # simulation clock, in milliseconds
KATHERINE = places.find("Katherine")
print(f"{KATHERINE.name}: {KATHERINE.latitude}, {KATHERINE.longitude}")
print(places.CREDIT)

## 1. Issue a signed warning

The warning console (on a Mac in the real app) writes a warning and **signs** it with the
publisher's private key. Phones only trust warnings signed by the one key built into the app.
This demo uses the public *development* key, so anyone can reproduce it; a real deployment
would use a key that never leaves the publisher.

The warning names the area it covers as up to four **area codes** (geohashes). A 5-character
code is a box about 5 km across.

In [ ]:
area = geohash.encode(KATHERINE.latitude, KATHERINE.longitude, 5)
draft = WarningDraft(
    hazard=HazardType.BUSHFIRE,
    severity=Severity.EMERGENCY_WARNING,
    headline="Bushfire near Katherine - leave now",
    action_text="Travel north on the Stuart Highway. Do not wait.",
    duration_hours=6,
    area_cells=[area],
)
print("Problems with the draft:", draft.problems or "none")

signer = OfficialAlertSigner()          # development key
warning = signer.sign(draft, new_alert_id(), START)
payload = wire.encode(warning)

print(f"Area code: {area}  ({places.label(area)})")
print(f"Signed warning: {len(payload)} bytes (must fit one Bluetooth frame: {wire.MAX_ENCODED_BYTES} bytes)")
print("First bytes:", payload[:24].hex(" "))
print("Signed by the key phones trust?", wire.verify_pinned(warning))

The whole warning fits in **one Bluetooth frame**, which matters on a crowded or patchy
network. The signature is the only thing a phone with no internet can check, and it is enough:
nobody without the publisher's key can make a warning that passes.

## 2. A forged or edited warning goes nowhere

Two attacks a prankster might try:

- **Forge** a warning with their own key ("Fake fire, drive south").
- **Edit** a real one: downgrade an Emergency Warning to Advice, or move its area.

Both fail the signature check. In the mesh, the first honest phone to receive one throws
it away and passes nothing on, even if the sender's phone has been modified to flood it.

In [ ]:
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from dataclasses import replace

attacker = OfficialAlertSigner(Ed25519PrivateKey.generate().private_bytes_raw())
forged = attacker.sign(replace(draft, headline="Fake fire - drive south now"), new_alert_id(), START)
downgraded = replace(warning, severity=Severity.ADVICE)
moved = replace(warning, area_cells=(geohash.encode(-12.4634, 130.8456, 5),))   # to Darwin

store = AlertStore(clock=lambda: START)
pd.DataFrame(
    [
        (name, wire.verify_pinned(item), store.ingest(item).value)
        for name, item in [("genuine", warning), ("forged", forged), ("downgraded", downgraded), ("moved", moved)]
    ],
    columns=["warning", "signature valid", "phone's store says"],
)

In [ ]:
# A line of 5 phones, 50 m apart. Phone 0 has been modified to flood a forgery anyway.
line = Mesh(start_ms=START)
phones = [line.add_phone(f"p{i}", *offset_m(KATHERINE.latitude, KATHERINE.longitude, 0, 50 * i)) for i in range(5)]
reached = line.broadcast(phones[0], OFFICIAL_ALERT_TYPE, wire.encode(forged))
print(f"Forgery reached {reached} phone(s); phones now holding it: {sum(bool(p.alert_store.live_alerts()) for p in phones)}")

line.send(phones[0], OFFICIAL_ALERT_TYPE, payload)
print(f"Genuine warning: phones holding it: {sum(bool(p.alert_store.live_alerts()) for p in phones)} of 5")

## 3. Watch the warning spread

A community of **300 phones** spread over about 1.2 km × 1.2 km around Katherine.
Only **1 in 10** has internet at the moment. A quarter of the people are walking about.

The warning is published at minute 0. Phones with internet get it at once and hand it to
the phones next to them over Bluetooth, which pass it on (up to 7 hops). After that,
phones swap what they hold whenever they meet, so people walking about carry it to
groups that were out of reach.

Press **▶** under the map to play it. Blue: warned over the internet. Red: warned over
Bluetooth. Grey: not warned yet. The whole map is inside the warning area.

In [ ]:
from alertmesh import viz

town = metrics.Scenario(n_phones=300, area_m=1200, share_online=0.10, share_moving=0.25, speed_mps=1.4, seed=4)
town_mesh, town_warning = metrics.build(town)
town_mesh.publish(wire.encode(town_warning))
frames = viz.spread_frames(town_mesh, town_warning.alert_id, minutes=30)

viz.spread_map(frames, title="Bushfire warning spreading through Katherine (simulation)")

In [ ]:
shares = viz.share_by_minute(frames).set_index("minute")["share warned"]
pd.DataFrame({"share of phones warned": shares.loc[[0, 1, 2, 5, 10, 15, 20, 30]].map("{:.0%}".format)})

Straight away about half the phones have it: the 1 in 10 with internet, plus everyone
within 7 Bluetooth hops of them. The rest arrive over the next minutes as people walk into
range of someone who already has it. Without the mesh, only the phones with internet would
ever be warned (section 8 compares the two).

## 4. Loud or quiet, and why

**Everyone hears about every warning**, but only a warning that covers you, at *Watch and Act*
or above, is **loud**. Everything else is a quiet notification. The phone decides using area
codes only; it never sends its location anywhere.

Six people, all with internet so each receives the warning. What differs is where they are
and what they let the app know:

In [ ]:
REASONS = {
    "insideArea": "You are inside the warning area",
    "adjacentToArea": "You are next to the warning area",
    "watchedPlaceInsideArea": "A place you watch is inside the warning area",
    "watchedPlaceNearArea": "A place you watch is next to the warning area",
    "outsideArea": "The warning is for another area",
    "locationUnknown": "Location unknown: told gently, never silenced",
    "lastKnownArea": "Your last known area is covered",
}

k_lat, k_lon = KATHERINE.latitude, KATHERINE.longitude
cell_south = geohash.decode_bounds(area)[0]      # southern edge of the warning area
people = Mesh(start_ms=START)
people.add_phone("Anna: at home in Katherine", k_lat, k_lon, has_internet=True)
people.add_phone("Ben: 60 km away", *offset_m(k_lat, k_lon, -60_000, 0), has_internet=True)
people.add_phone("Fin: just outside the area", cell_south - 0.001, k_lon, has_internet=True)
cara = people.add_phone("Cara: location off, was in Katherine earlier", k_lat, k_lon, has_internet=True)
cara.location_on = False                                   # keeps the rough area remembered
dev = people.add_phone("Dev: location off, watches Katherine", *offset_m(k_lat, k_lon, -60_000, 0), has_internet=True)
dev.location_on, dev.remembered_cell, dev.bookmarks = False, None, (area,)
eve = people.add_phone("Eve: location off, nothing remembered", k_lat, k_lon, has_internet=True)
eve.location_on, eve.remembered_cell = False, None

people.publish(payload)
pd.DataFrame(
    [(p.id, p.notifications[0].urgency.name.lower(), REASONS[p.notifications[0].reason.kind.value])
     for p in people.phones.values()],
    columns=["person", "notification", "why"],
).set_index("person")

Cara and Dev chose not to share their location, and are still warned loudly: the app
remembers Cara's rough area (about 40 km), and Dev asked to watch Katherine. Eve's phone
knows nothing, so it tells her gently rather than guess. Being too loud is the safer mistake,
but silence is never an option.

Across the whole town from section 3, every warned phone inside the area was told loudly:

In [ ]:
def level(phone):
    urgency = phone.loudest(town_warning.alert_id)
    return "not warned yet" if urgency is proximity.Urgency.SILENT else urgency.name.lower()

town_levels = pd.Series([level(p) for p in town_mesh.phones.values()]).value_counts()
town_levels.rename("phones").to_frame()

## 5. Update, then cancel

Real warnings change. A flood *Watch and Act* becomes an *Emergency Warning*, and later the
warning is withdrawn. Every update keeps the same event ID with a later issue time, so phones
**replace** the old version instead of showing two; a withdrawal removes it everywhere it
has travelled, and keeps travelling itself so it outruns old copies still on the move.

Same town as section 3, continuing from where it was:

In [ ]:
def stage(label, event_id, level=None):
    """Phones showing this event (any version), and how many show it at `level`."""
    # A phone's store holds one version per event, so this is one entry per phone.
    shown = [a for p in town_mesh.phones.values() for a in p.alert_store.live_alerts() if a.alert_id == event_id]
    at_level = [a for a in shown if a.severity is level]
    return {"stage": label, "phones showing the warning": len(shown),
            "showing this version": len(at_level) if level else "-",
            "notifications so far": sum(n.alert_id == event_id for p in town_mesh.phones.values() for n in p.notifications)}

flood = WarningDraft(HazardType.FLOOD, Severity.WATCH_AND_ACT, "Katherine River rising - prepare to leave",
                     "Move vehicles and animals to high ground.", 12, [area])
flood_v1 = signer.sign(flood, new_alert_id(), town_mesh.now_ms)
town_mesh.publish(wire.encode(flood_v1))
town_mesh.run(10 * 60)
rows = [stage("Watch and Act, after 10 min", flood_v1.alert_id, Severity.WATCH_AND_ACT)]

update = WarningDraft.updating(flood_v1, town_mesh.now_ms)
update.severity, update.headline = Severity.EMERGENCY_WARNING, "Katherine River flooding - leave now"
flood_v2 = signer.sign(update, flood_v1.alert_id, town_mesh.now_ms)      # same event, later version
town_mesh.publish(wire.encode(flood_v2))
town_mesh.run(10 * 60)
rows.append(stage("updated to Emergency Warning, after 10 min", flood_v1.alert_id, Severity.EMERGENCY_WARNING))

town_mesh.publish(wire.encode(signer.cancel(flood_v1.alert_id, town_mesh.now_ms)))
town_mesh.run(10 * 60)
rows.append(stage("withdrawn, after 10 min", flood_v1.alert_id))
pd.DataFrame(rows).set_index("stage")

In [ ]:
# An old copy still travelling on the mesh arrives late. Phones refuse it: the withdrawal wins.
someone = town_mesh.phones["p42"]
print("Old Watch and Act copy arrives:", someone.alert_store.ingest(flood_v1).value)
print("Old Emergency Warning copy arrives:", someone.alert_store.ingest(flood_v2).value)

At no point does anyone see two versions of the same flood. The update made phones notify
again (it is louder news). The withdrawal spreads the same way the warning did: within
10 minutes it had cleared the flood from almost every phone, including ones that only ever
heard about it over Bluetooth. The few still showing it are out of reach for now; they clear
it when they next meet someone who has the withdrawal, or when the warning expires.

## 6. Call for help, then "I'm safe"

Help flows the other way too. Someone with no signal presses **SOS**. Their phone signs the
call with its own key and sends it over Bluetooth; it travels and is carried like a warning.
The location is cut to about 150 m: enough to find a person, not enough to pinpoint a home.
Phones near the caller are told **loudly**, others quietly.

In [ ]:
caller = town_mesh.phones["p7"]
caller.nickname = caller.author.nickname = "Sam"
sos = town_mesh.send_sos(caller, "Car stuck at the causeway, water rising")
print(f"SOS from {sos.author_nickname}: '{sos.note}' at {sos.geohash} ({places.label(sos.geohash)})")

def holding(report):
    """Phones whose current version of this person's record is this kind (SOS or safe).
    An "I'm safe" reuses the SOS's ID and replaces it, so the kind tells them apart."""
    key = (report.author_signing_key, report.report_id)
    return sum(any((r.author_signing_key, r.report_id) == key and r.kind is report.kind
                   for r in p.report_store.live_reports())
               for p in town_mesh.phones.values())

rows = [{"minutes after SOS": 0, "phones holding it": holding(sos)}]
for minute in (2, 5, 10):
    town_mesh.run((minute - rows[-1]["minutes after SOS"]) * 60)
    rows.append({"minutes after SOS": minute, "phones holding it": holding(sos)})
told = pd.Series([n.urgency.name.lower() for p in town_mesh.phones.values()
                  for n in p.report_notifications if n.kind is reports.ReportKind.SOS]).value_counts()
display(pd.DataFrame(rows).set_index("minutes after SOS"))
told.rename("SOS notifications").to_frame()

**Can a stranger call off someone else's SOS?** That would be the most harmful prank of all:
rescuers stop coming. Two tries:

1. Send an "I'm safe" *pretending to be Sam* (Sam's key, but not signed by Sam).
2. Send an "I'm safe" with *their own* key, reusing the ID of Sam's SOS.

In [ ]:
stranger_phone = town_mesh.phones["p99"]
stranger_key = Ed25519PrivateKey.generate()

def fake_safe(claimed_key, signing_key):
    fields = (reports.ReportKind.SAFE, sos.report_id, sos.geohash, 0, None, "Sam is fine, go home",
              claimed_key, "Sam", town_mesh.now_ms, town_mesh.now_ms + 60 * 60 * 1000)
    return reports.CommunityReport(*fields, signing_key.sign(reports.report_signing_bytes(*fields)))

pretending = fake_safe(sos.author_signing_key, stranger_key)                       # try 1
own_key = fake_safe(stranger_key.public_key().public_bytes_raw(), stranger_key)    # try 2
for label, fake in (("pretending to be Sam", pretending), ("own key, Sam's SOS ID", own_key)):
    town_mesh.broadcast(stranger_phone, COMMUNITY_REPORT_TYPE, reports.encode(fake))
town_mesh.run(5 * 60)
print("Phones still holding Sam's SOS:", holding(sos))

Try 1 fails the signature check and goes nowhere. Try 2 is a valid report, but from a
different person, so it is stored as the stranger's own check-in and **does not replace
Sam's SOS**. Only Sam can call off Sam's SOS:

In [ ]:
safe = town_mesh.send_safe(caller, "Rescued by the SES, thank you")
town_mesh.run(10 * 60)
print("Phones still holding Sam's SOS:", holding(sos))
print("Phones holding Sam's 'I'm safe':", holding(safe))
told_safe = sum(1 for p in town_mesh.phones.values() for n in p.report_notifications
                if n.kind is reports.ReportKind.SAFE and n.nickname == "Sam")
print("Phones told that Sam is safe:", told_safe)

Sam's SOS started on a phone with no signal and reached about 24 phones at once, then
spread as people moved. The "I'm safe" travels the same way and at the same pace, so for
a while some phones still show the SOS; each one switches to "Sam is safe" as soon as the
update reaches it. Nobody else can make that switch happen.